In [1]:
from pathlib import Path

import pandas as pd
import numpy as np

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier

import joblib

In [2]:
# 自动定位项目根目录，兼容从notebook/或项目根目录运行
CURRENT_DIR = Path.cwd()

if (CURRENT_DIR / "data").exists() and (CURRENT_DIR / "notebook").exists():
    PROJECT_DIR = CURRENT_DIR
elif (CURRENT_DIR.parent / "data").exists() and (
    (CURRENT_DIR.parent / "notebook").exists() or (CURRENT_DIR.parent / "notebooks").exists()
):
    PROJECT_DIR = CURRENT_DIR.parent
else:
    raise FileNotFoundError(
        "Cannot locate project root. Please check whether you are inside the project folder."
    )

DATA_RAW_DIR = PROJECT_DIR / "data" / "raw"
SUBMISSION_DIR = PROJECT_DIR / "submissions"
MODEL_DIR = PROJECT_DIR / "models"

SUBMISSION_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

print("Project directory:", PROJECT_DIR)
print("Raw data directory:", DATA_RAW_DIR)
print("Submission directory:", SUBMISSION_DIR)
print("Model directory:", MODEL_DIR)

Project directory: d:\A_projects\kaggle-star-type-prediction
Raw data directory: d:\A_projects\kaggle-star-type-prediction\data\raw
Submission directory: d:\A_projects\kaggle-star-type-prediction\submissions
Model directory: d:\A_projects\kaggle-star-type-prediction\models


In [3]:
#读取数据
train_path = DATA_RAW_DIR / "train.csv"
test_path = DATA_RAW_DIR / "test.csv"
sample_submission_path = DATA_RAW_DIR / "sample_submission.csv"

train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)
sample_submission = pd.read_csv(sample_submission_path)

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)
print("Sample submission shape:", sample_submission.shape)

display(train_df.head())
display(test_df.head())
display(sample_submission.head())

Train shape: (577347, 12)
Test shape: (247435, 11)
Sample submission shape: (247435, 2)


,id,alpha,delta,u,g,r,i,z,redshift,spectral_type,galaxy_population,class
0,0,147.734256,16.959273,25.472123,21.895559,20.357926,19.257113,18.621057,0.408982,M,Red_Sequence,GALAXY
1,1,127.988677,32.346716,20.778509,19.087062,17.587208,17.226067,16.786433,0.157976,M,Red_Sequence,GALAXY
2,2,179.792648,35.344843,21.035203,21.079128,21.171840,20.582629,20.557366,2.823770,O/B,Blue_Cloud,QSO
3,3,225.818295,48.569421,23.305056,21.050736,19.017754,18.365658,17.914952,0.536099,M,Red_Sequence,GALAXY
4,4,141.836135,19.342852,21.703158,19.471680,18.234449,17.899447,17.616185,0.555761,M,Red_Sequence,GALAXY


,id,alpha,delta,u,g,r,i,z,redshift,spectral_type,galaxy_population
0,577347,120.719779,23.924249,23.668066,21.951680,21.086183,20.180032,19.202124,0.429042,G/K,Red_Sequence
1,577348,219.414419,42.171651,24.902933,22.338822,20.732163,19.860330,19.687691,0.867305,M,Red_Sequence
2,577349,173.568731,-1.756400,19.427591,18.474633,17.551314,16.570674,16.176765,0.224234,G/K,Blue_Cloud
3,577350,184.903993,-1.411074,23.121029,21.526855,20.670159,20.417633,20.699095,0.066507,G/K,Red_Sequence
4,577351,222.487816,15.381403,25.094282,22.643981,21.123173,19.439500,19.094158,0.977218,M,Red_Sequence


,id,class
0,577347,GALAXY
1,577348,GALAXY
2,577349,GALAXY
3,577350,GALAXY
4,577351,GALAXY


In [4]:
#确认目标列和ID列
TARGET_COL = "class"
ID_COL = "id"

if TARGET_COL not in train_df.columns:
    raise ValueError(f"Target column '{TARGET_COL}' not found in train.csv.")

if ID_COL not in test_df.columns:
    raise ValueError(f"ID column '{ID_COL}' not found in test.csv.")

print("Target distribution:")
display(train_df[TARGET_COL].value_counts())

Target distribution:


class
GALAXY    377480
QSO       117143
STAR       82724
Name: count, dtype: int64

In [5]:
#构造颜色指数特征
def add_color_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    Add astronomical color index features.
    """
    df = df.copy()
    
    required_cols = ["u", "g", "r", "i", "z"]
    missing_cols = [col for col in required_cols if col not in df.columns]
    
    if missing_cols:
        raise ValueError(f"Missing columns for color features: {missing_cols}")
    
    df["u_g"] = df["u"] - df["g"]
    df["g_r"] = df["g"] - df["r"]
    df["r_i"] = df["r"] - df["i"]
    df["i_z"] = df["i"] - df["z"]
    
    return df


train_fe = add_color_features(train_df)
test_fe = add_color_features(test_df)

display(train_fe.head())
display(test_fe.head())

,id,alpha,delta,u,g,r,i,z,redshift,spectral_type,galaxy_population,class,u_g,g_r,r_i,i_z
0,0,147.734256,16.959273,25.472123,21.895559,20.357926,19.257113,18.621057,0.408982,M,Red_Sequence,GALAXY,3.576564,1.537632,1.100813,0.636056
1,1,127.988677,32.346716,20.778509,19.087062,17.587208,17.226067,16.786433,0.157976,M,Red_Sequence,GALAXY,1.691447,1.499854,0.361141,0.439634
2,2,179.792648,35.344843,21.035203,21.079128,21.171840,20.582629,20.557366,2.823770,O/B,Blue_Cloud,QSO,-0.043925,-0.092712,0.589211,0.025263
3,3,225.818295,48.569421,23.305056,21.050736,19.017754,18.365658,17.914952,0.536099,M,Red_Sequence,GALAXY,2.254320,2.032982,0.652096,0.450706
4,4,141.836135,19.342852,21.703158,19.471680,18.234449,17.899447,17.616185,0.555761,M,Red_Sequence,GALAXY,2.231478,1.237231,0.335002,0.283262


,id,alpha,delta,u,g,r,i,z,redshift,spectral_type,galaxy_population,u_g,g_r,r_i,i_z
0,577347,120.719779,23.924249,23.668066,21.951680,21.086183,20.180032,19.202124,0.429042,G/K,Red_Sequence,1.716387,0.865497,0.906151,0.977908
1,577348,219.414419,42.171651,24.902933,22.338822,20.732163,19.860330,19.687691,0.867305,M,Red_Sequence,2.564112,1.606658,0.871833,0.172640
2,577349,173.568731,-1.756400,19.427591,18.474633,17.551314,16.570674,16.176765,0.224234,G/K,Blue_Cloud,0.952958,0.923319,0.980640,0.393909
3,577350,184.903993,-1.411074,23.121029,21.526855,20.670159,20.417633,20.699095,0.066507,G/K,Red_Sequence,1.594174,0.856696,0.252526,-0.281462
4,577351,222.487816,15.381403,25.094282,22.643981,21.123173,19.439500,19.094158,0.977218,M,Red_Sequence,2.450301,1.520808,1.683673,0.345342


In [6]:
#确定最终特征列
BASE_FEATURES = ["u", "g", "r", "i", "z", "redshift", "alpha", "delta"]
COLOR_FEATURES = ["u_g", "g_r", "r_i", "i_z"]

FEATURE_COLS = BASE_FEATURES + COLOR_FEATURES

missing_train_cols = [col for col in FEATURE_COLS if col not in train_fe.columns]
missing_test_cols = [col for col in FEATURE_COLS if col not in test_fe.columns]

if missing_train_cols:
    raise ValueError(f"Missing feature columns in train data: {missing_train_cols}")

if missing_test_cols:
    raise ValueError(f"Missing feature columns in test data: {missing_test_cols}")

X_train_full = train_fe[FEATURE_COLS].copy()
y_train_full = train_fe[TARGET_COL].copy()
X_test = test_fe[FEATURE_COLS].copy()

print("Final feature columns:")
print(FEATURE_COLS)

print("X_train_full shape:", X_train_full.shape)
print("X_test shape:", X_test.shape)

Final feature columns:
['u', 'g', 'r', 'i', 'z', 'redshift', 'alpha', 'delta', 'u_g', 'g_r', 'r_i', 'i_z']
X_train_full shape: (577347, 12)
X_test shape: (247435, 12)


In [7]:
#训练最终模型
final_model = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        (
            "model",
            RandomForestClassifier(
                n_estimators=300,
                random_state=42,
                n_jobs=-1
            )
        )
    ]
)

final_model.fit(X_train_full, y_train_full)

print("Final RandomForest + color features model has been trained on the full training dataset.")

Final RandomForest + color features model has been trained on the full training dataset.


In [8]:
#生成测试集预测
test_pred = final_model.predict(X_test)

submission_df = sample_submission.copy()

if TARGET_COL in submission_df.columns:
    submission_df[TARGET_COL] = test_pred
else:
    # 如果sample_submission的目标列不是class，则自动使用第二列作为预测列
    pred_col = submission_df.columns[1]
    submission_df[pred_col] = test_pred
    print(f"Prediction column is '{pred_col}', not '{TARGET_COL}'.")

display(submission_df.head())

print("Prediction distribution:")
display(pd.Series(test_pred).value_counts())

,id,class
0,577347,GALAXY
1,577348,GALAXY
2,577349,GALAXY
3,577350,STAR
4,577351,GALAXY


Prediction distribution:


GALAXY    162149
QSO        50155
STAR       35131
Name: count, dtype: int64

In [9]:
#保存提交文件和模型
submission_path = SUBMISSION_DIR / "random_forest_color_submission.csv"
model_path = MODEL_DIR / "random_forest_color_full_model.pkl"

submission_df.to_csv(submission_path, index=False)
joblib.dump(final_model, model_path)

print("Submission file saved to:")
print(submission_path)

print("Model file saved to:")
print(model_path)

Submission file saved to:
d:\A_projects\kaggle-star-type-prediction\submissions\random_forest_color_submission.csv
Model file saved to:
d:\A_projects\kaggle-star-type-prediction\models\random_forest_color_full_model.pkl


In [10]:
#检查提交文件格式
check_submission = pd.read_csv(submission_path)

print("Submission shape:", check_submission.shape)
display(check_submission.head())

print("Missing values:")
display(check_submission.isna().sum())

print("Columns:")
print(check_submission.columns.tolist())

Submission shape: (247435, 2)


,id,class
0,577347,GALAXY
1,577348,GALAXY
2,577349,GALAXY
3,577350,STAR
4,577351,GALAXY


Missing values:


id       0
class    0
dtype: int64

Columns:
['id', 'class']
